# Teacher Neural State Comparison

Compare healthy vs ablated teacher neural states across all 14 memory neurons.

This notebook:
1. Loads extracted neural states from healthy and ablated teachers
2. Performs direct state comparisons (L2 distances)
3. Applies PCA for dimensionality reduction and visualization
4. Ranks all 14 neurons by ablation impact
5. Provides side-by-side comparisons showing how ablating each neuron affects the neural state geometry

**Prerequisites:** Must first run `extract_all_teacher_states.sh` to generate the teacher state files.

## Section 1: Setup and Imports

Import required libraries and define paths to data directories.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA

# Set up paths
REPO_ROOT = Path('/scratch3/shaiq_home/repos/behaviour_ddpm')
DATA_DIR = REPO_ROOT / 'teacher_neural_states'
RESULTS_DIR = REPO_ROOT / 'ddpm' / 'analysis' / 'new_analysis' / 'results'

# Create results directory if it doesn't exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"\nChecking for teacher state files...")

# List available teacher state files
if DATA_DIR.exists():
    files = sorted(DATA_DIR.glob('*.npz'))
    print(f"Found {len(files)} teacher state files")
    for f in files[:3]:
        print(f"  - {f.name}")
    if len(files) > 3:
        print(f"  ... and {len(files)-3} more")
else:
    print(f"⚠️  Data directory not found. Run extract_all_teacher_states.sh first!")

## Section 2: Load Teacher States

Load healthy baseline and ablated teacher neural states from extracted .npz files.

In [ ]:
print("="*70)
print("LOADING TEACHER NEURAL STATES FOR COMPARISON")
print("="*70)

# Load healthy baseline
healthy_path = DATA_DIR / "healthy_teacher.npz"
healthy_data = np.load(healthy_path)
healthy_states = healthy_data['neural_states']
healthy_metadata = healthy_data['metadata']

print(f"\n✓ Healthy teacher loaded")
print(f"  States shape: {healthy_states.shape}")
print(f"  Mean: {healthy_states.mean():.4f}, Std: {healthy_states.std():.4f}")

# Load ablated teacher (neuron 0 as example)
ablated_neuron = 0
ablated_path = DATA_DIR / f"ablated_teacher_neuron_{ablated_neuron}.npz"
ablated_data = np.load(ablated_path)
ablated_states = ablated_data['neural_states']
ablated_metadata = ablated_data['metadata']

print(f"\n✓ Ablated teacher loaded (neuron {ablated_neuron})")
print(f"  States shape: {ablated_states.shape}")
print(f"  Mean: {ablated_states.mean():.4f}, Std: {ablated_states.std():.4f}")

# Verify metadata matches
assert np.allclose(healthy_metadata, ablated_metadata), "Metadata mismatch!"
print(f"\n✓ Metadata verified - same trials in both")

## Section 3: Direct State Comparison

Compute L2 distances and analyze per-dimension changes between healthy and ablated states.

In [ ]:
print("\n" + "="*70)
print("COMPARING HEALTHY VS ABLATED TEACHER STATES")
print("="*70)

# Compute difference
state_diff = healthy_states - ablated_states

print(f"\nState differences:")
print(f"  L2 distance (mean): {np.linalg.norm(state_diff, axis=1).mean():.4f}")
print(f"  L2 distance (std): {np.linalg.norm(state_diff, axis=1).std():.4f}")
print(f"  Max abs difference: {np.abs(state_diff).max():.4f}")

# Per-dimension changes
dim_changes = np.abs(state_diff).mean(axis=0)
print(f"\nPer-dimension mean absolute changes:")
for dim in range(16):
    marker = " ← ABLATED" if dim == ablated_neuron else ""
    print(f"  Dim {dim:2d}: {dim_changes[dim]:8.4f}{marker}")

# Separate by cue
cue1_mask = healthy_metadata[:, 0] == 1
cue2_mask = healthy_metadata[:, 0] == 2

healthy_cue1_mean = healthy_states[cue1_mask].mean(axis=0)
healthy_cue2_mean = healthy_states[cue2_mask].mean(axis=0)
ablated_cue1_mean = ablated_states[cue1_mask].mean(axis=0)
ablated_cue2_mean = ablated_states[cue2_mask].mean(axis=0)

print(f"\nCue-specific changes:")
print(f"  Healthy cue separation: {np.linalg.norm(healthy_cue1_mean - healthy_cue2_mean):.4f}")
print(f"  Ablated cue separation: {np.linalg.norm(ablated_cue1_mean - ablated_cue2_mean):.4f}")

## Section 4: PCA Analysis Helper Function

Define helper function to bin neural states by cued color and apply PCA transformation.

In [ ]:
def bin_angle(angle, bin_size=30.0):
    """Bin an angle into discrete bins."""
    return int(angle // bin_size) % int(360 // bin_size)


def bin_and_average_by_cued_color(states, metadata, n_bins=12):
    """Bin trials by cued color and average neural states."""
    bin_size = 360.0 / n_bins
    
    binned_data = {
        1: {b: [] for b in range(n_bins)},
        2: {b: [] for b in range(n_bins)}
    }
    
    for i, (cue, c1, c2) in enumerate(metadata):
        cue = int(cue)
        cued_angle = c1 if cue == 1 else c2
        bin_idx = bin_angle(cued_angle, bin_size)
        binned_data[cue][bin_idx].append(states[i])
    
    averaged_data = {
        1: np.zeros((n_bins, states.shape[1])),
        2: np.zeros((n_bins, states.shape[1]))
    }
    
    for cue in [1, 2]:
        for b in range(n_bins):
            if len(binned_data[cue][b]) > 0:
                averaged_data[cue][b] = np.mean(binned_data[cue][b], axis=0)
            else:
                averaged_data[cue][b] = np.nan
    
    return averaged_data, binned_data


def bin_and_pca_teacher_states(states, metadata, n_bins=12):
    """Apply binning and PCA to teacher states."""
    averaged_states, _ = bin_and_average_by_cued_color(states, metadata, n_bins)
    
    all_binned = np.vstack([averaged_states[1], averaged_states[2]])
    valid_mask = ~np.isnan(all_binned).any(axis=1)
    all_binned_clean = all_binned[valid_mask]
    
    pca = PCA(n_components=3)
    pca_coords = pca.fit_transform(all_binned_clean)
    
    cue1_pca = pca_coords[:n_bins]
    cue2_pca = pca_coords[n_bins:2*n_bins]
    
    return pca, pca_coords, cue1_pca, cue2_pca, all_binned_clean

print("✓ PCA helper functions defined")

## Section 5: Apply PCA to Both Teachers

Run PCA analysis on healthy and ablated teacher states separately.

In [ ]:
print("\n" + "="*70)
print("PCA COMPARISON: HEALTHY VS ABLATED")
print("="*70)

# PCA for healthy
pca_h, coords_h, cue1_h, cue2_h, binned_h = bin_and_pca_teacher_states(healthy_states, healthy_metadata)
print(f"\nHealthy teacher PCA:")
print(f"  Explained variance: {pca_h.explained_variance_ratio_}")
print(f"  Cue separation (PC1-PC2): {np.linalg.norm(cue1_h.mean(axis=0) - cue2_h.mean(axis=0)):.4f}")

# PCA for ablated
pca_a, coords_a, cue1_a, cue2_a, binned_a = bin_and_pca_teacher_states(ablated_states, ablated_metadata)
print(f"\nAblated teacher PCA:")
print(f"  Explained variance: {pca_a.explained_variance_ratio_}")
print(f"  Cue separation (PC1-PC2): {np.linalg.norm(cue1_a.mean(axis=0) - cue2_a.mean(axis=0)):.4f}")

## Section 6: Side-by-Side Visualization

Create side-by-side scatter plots comparing healthy and ablated teacher states in PC space.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot healthy
ax = axes[0]
ax.scatter(cue1_h[:, 0], cue1_h[:, 1], c='red', alpha=0.6, s=100, label='Cue 1')
ax.scatter(cue2_h[:, 0], cue2_h[:, 1], c='blue', alpha=0.6, s=100, label='Cue 2')
ax.set_xlabel(f'PC1 ({pca_h.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_h.explained_variance_ratio_[1]:.1%})')
ax.set_title('Healthy Teacher (Baseline)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axis('equal')

# Plot ablated
ax = axes[1]
ax.scatter(cue1_a[:, 0], cue1_a[:, 1], c='red', alpha=0.6, s=100, label='Cue 1')
ax.scatter(cue2_a[:, 0], cue2_a[:, 1], c='blue', alpha=0.6, s=100, label='Cue 2')
ax.set_xlabel(f'PC1 ({pca_a.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_a.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Ablated Teacher (Neuron {ablated_neuron})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axis('equal')

plt.tight_layout()
plt.savefig(RESULTS_DIR / f'teacher_comparison_neuron_{ablated_neuron}.png', dpi=150)
plt.show()

print(f"\n✓ Saved comparison plot to {RESULTS_DIR / f'teacher_comparison_neuron_{ablated_neuron}.png'}")

## Section 7: Load All Ablated Teachers

Load neural states for all 14 ablated teachers for comprehensive comparison.

In [ ]:
print("\n" + "="*70)
print("LOADING ALL 14 ABLATED TEACHERS")
print("="*70)

all_teachers = {}
all_teachers['healthy'] = {
    'states': healthy_states,
    'metadata': healthy_metadata,
    'neuron': None
}

for neuron in range(14):
    path = DATA_DIR / f"ablated_teacher_neuron_{neuron}.npz"
    if path.exists():
        data = np.load(path)
        all_teachers[f'neuron_{neuron}'] = {
            'states': data['neural_states'],
            'metadata': data['metadata'],
            'neuron': neuron
        }
        print(f"  ✓ Loaded neuron {neuron}")
    else:
        print(f"  ✗ Could not find neuron {neuron}")

print(f"\n✓ Loaded {len(all_teachers)} teachers (1 healthy + {len(all_teachers)-1} ablated)")

## Section 8: Compute Ablation Impact Rankings

Calculate L2 distances between healthy and each ablated teacher, ranking neurons by impact.

In [ ]:
print("\n" + "="*70)
print("COMPUTING ABLATION IMPACT")
print("="*70)

healthy_states_ref = all_teachers['healthy']['states']
distances = {}

for name, teacher in all_teachers.items():
    if name == 'healthy':
        continue
    diff = np.linalg.norm(healthy_states_ref - teacher['states'], axis=1).mean()
    distances[name] = diff

# Sort by impact
sorted_neurons = sorted(distances.items(), key=lambda x: x[1], reverse=True)

print(f"\nNeurons ranked by ablation impact (L2 distance from healthy):")
for i, (name, dist) in enumerate(sorted_neurons, 1):
    neuron_idx = int(name.split('_')[1])
    print(f"  {i:2d}. Neuron {neuron_idx:2d}: {dist:8.4f}")

## Section 9: Visualize Ablation Impact

Create bar plot ranking neurons by ablation impact magnitude.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Extract neurons and distances
neurons = [int(name.split('_')[1]) for name, _ in sorted_neurons]
dists = [d for _, d in sorted_neurons]

# Create bar plot
bars = ax.bar(range(len(neurons)), dists, color='steelblue', alpha=0.7)

# Color the bars in order
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(neurons)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax.set_xticks(range(len(neurons)))
ax.set_xticklabels([str(n) for n in neurons])
ax.set_xlabel('Ablated Neuron (sorted by impact)', fontsize=12)
ax.set_ylabel('Mean L2 Distance from Healthy', fontsize=12)
ax.set_title('Impact of Ablating Each Memory Neuron on Neural State', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, dist) in enumerate(zip(bars, dists)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{dist:.2f}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_impact_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✓ Saved ranking plot to {RESULTS_DIR / 'ablation_impact_ranking.png'}")
print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)